### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\choud\AppData\Local\Temp\ipykernel_19192\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\choud\OneDrive\Desktop\RAG_00\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: attention.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/EKHOMP+AdvOT830b2708.B', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 46, '/FontDescriptor': IndirectObject(158, 0, 1793691644816), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [270, 0, 0, 552, 552, 552, 552, 552, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 718, 0, 0, 604, 0, 0, 0, 0, 0, 604, 822, 0, 0, 666, 0, 718, 666, 604, 0, 0, 937, 0, 0, 0, 0, 0, 0, 0, 0, 0, 552, 0, 552, 604, 552, 0, 0, 604, 270, 0, 0, 270, 885, 604, 604, 604, 0, 385, 552, 322, 604, 0, 0, 0, 552]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/EKHOKN+AdvOTa20b42a7', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 40, '/FontDescriptor': IndirectObject(160, 0, 1793691644816), '/LastChar': 239, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [322,

  ✓ Loaded 15 pages

Processing: PM287.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/EIPJIN+AdvTTd0b5fdba.I', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 58, '/FontDescriptor': IndirectObject(79, 0, 1793691644816), '/LastChar': 61, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [270, 0, 0, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/EKHONP+AdvOT5e566946.I', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 49, '/FontDescriptor': IndirectObject(75, 0, 1793691644816), '/LastChar': 117, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [552, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 718, 0, 0, 604, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 718, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 552, 0, 500, 552, 552, 0, 0, 552, 218, 0, 0, 0, 822, 552, 552, 552, 0, 322, 500, 270, 552]}, but is not installed. Consider ins

  ✓ Loaded 21 pages

Total documents loaded: 36


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of To

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 36 documents into 108 chunks

Example chunk:
Content: Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of To

### embedding and vectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2168.90it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\choud\AppData\Local\Temp\ipykernel_19192\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 324


In [9]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of To

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 108 texts...


Batches: 100%|██████████| 4/4 [00:04<00:00,  1.05s/it]


Generated embeddings with shape: (108, 384)
Adding 108 documents to vector store...
Successfully added 108 documents to vector store
Total documents in collection: 432


### Retriever Pipeline from VectorStore

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.39it/s]

Generated embeddings with shape: (1, 384)
Retrieved 4 documents (after filtering)


[{'id': 'doc_f2ebc495_12',
  'content': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3',
  'metadata': {'trapped': '/False',
   'page_label': '3',
   'content_length': 216,
   'creationdate': '2024-04-10T21:11:43+00:00',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'creator': 'LaTeX with hyperref',
   'doc_index': 12,
   'page': 2,
   'moddate': '2024-04-10T21:11:43+00:00',
   'title': '',
   'author': '',
   'keywords': '',
   'total_pages': 15,
   'producer': 'pdfTeX-1.40.25',
   'source': '..\\data\\pdf\\attention.pdf',
   'source_file': 'attention.pdf',
   'file_type': 'pdf',
   'subject': ''},
  'similarity_score': 0.13995468616485596,
  'distance': 0.860045313835144,
  'rank': 1},
 {'id': 'doc_0f8dee02_12',
  'content': '3.2 Attention\nAn a

In [14]:
rag_retriever.retrieve("3 assumptions in poisson process")

Retrieving documents for query: '3 assumptions in poisson process'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.49it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_955a644a_76',
  'content': 'lt þ o(t).\n(iii) The probability of two or more arrivals in an interval of length t is o(t).\nThe parameter l is constant over all t intervals, and o(t) represents any function\ng(t) for which lim t→0 g(t)/t ¼ 0. Condition (ii) says that, for infinitesimal t,t h e\nprobability of exactly one arrival in an interval of length t is lt plus a quantity\nvery small in comparison tot, and condition (iii) says that, for infinitesimalt,t h e\nprobability of two or more arrivals in an interval of length t is very small in\ncomparison to t. The random time points are called Poisson points, and each\nrealization of the Poisson process corresponds to a set of time points result-\ning from a particular observation of the arrival process. It can be proven that\nX(t) possesses a Poisson density with mean and variance equal to lt, namely,\nPðX ðtÞ¼ kÞ¼ e/C0 lt ðltÞk\nk! (1.33)\nfor k ¼ 0, 1, 2, ::: .\nIt follows from assumption (i) that the Poisson process has 

In [15]:
rag_retriever.retrieve("number of  assumptions in poisson process")

Retrieving documents for query: 'number of  assumptions in poisson process'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.43it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_9223ed06_94',
  'content': 'KX ðt, t0Þ¼ KX ðt /C0 t0, t0 /C0 t0Þ¼ KX ðt /C0 t0,0 Þ, (1.58)\nwhich is a function of t /C0 t0.\nExample 1.2. Let Y(t) be the Poisson process with mean lt and r be a positive\nconstant. The Poisson increment process is defined by\nX ðtÞ¼ Y ðt þ rÞ/C0 Y ðtÞ:\nAccording to the Poisson model, X(t) counts the number of points in\n[t, t þ r], and\nmX ðtÞ¼ E½Y ðt þ rÞ/C138 /C0 E½Y ðtÞ/C138 ¼ lr:\nFor the covariance KX (t, t0), there are two cases: | t /C0 t0|> r and | t /C0 t0| ≤ r.\nIf |t /C0 t0|> r, then the intervals determined by t and t0 are nonoverlapping and,\nowing to independent increments, X(t)a n d X(t0) are independent, and their\ncovariance is 0. Suppose that | t /C0 t0| ≤ r. First consider the case where t < t0.\nThen t < t0 < t þ r < t0 þ r, and we can apply the result of Eq. 1.34 together with\nthe observation that, because the process counts the number of points in\n[t, t þ r], its mean is lr.T h u s ,\n15Random Functions',
  'metada

In [16]:
rag_retriever.retrieve("how many assumptions in poisson process")

Retrieving documents for query: 'how many assumptions in poisson process'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.91it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_461a3cc4_81',
  'content': 'The Poisson process is generated by random Poisson points, which are\noften said to model complete randomness, meaning that intuitively they\nmodel a “uniform” distribution of points across the infinite interval [0, `). For\napplication, a basic proposition states that the time distribution governing the\narrival of the kth Poisson point following a given Poisson point is governed\nby a gamma distribution with a ¼ k and b ¼ 1/l (lt being the mean of the\nPoisson process). In particular, for k ¼ 1, the inter-arrival time is governed by\nan exponential distribution with mean 1/ l.\nUp to this point we have based our discussion of the Poisson process on the\nconditions defining a Poisson arrival process, a key consequence being that the\nprocess possesses independent increments. In general, a random processX(t), t ≥ 0,\nis said to have independent incrementsif X(0) ¼ 0 and, for any t1 , t2 , ··· , tn,\nthe random variables X ðt2Þ/C0 X ðt1Þ, X ðt3Þ/

### RAG Pipeline- VectorDB To LLM Output Generation

In [17]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.is_file():
    env_path = Path.cwd().parent / ".env"

load_dotenv(env_path)
print(f"GROQ_API_KEY loaded: {bool(os.getenv('GROQ_API_KEY'))}")

GROQ_API_KEY loaded: True


In [18]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage


In [19]:
class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-20b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [20]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-20b
Groq LLM initialized successfully!


In [21]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("what is poisson process")

Retrieving documents for query: 'what is poisson process'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.11it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_461a3cc4_81',
  'content': 'The Poisson process is generated by random Poisson points, which are\noften said to model complete randomness, meaning that intuitively they\nmodel a “uniform” distribution of points across the infinite interval [0, `). For\napplication, a basic proposition states that the time distribution governing the\narrival of the kth Poisson point following a given Poisson point is governed\nby a gamma distribution with a ¼ k and b ¼ 1/l (lt being the mean of the\nPoisson process). In particular, for k ¼ 1, the inter-arrival time is governed by\nan exponential distribution with mean 1/ l.\nUp to this point we have based our discussion of the Poisson process on the\nconditions defining a Poisson arrival process, a key consequence being that the\nprocess possesses independent increments. In general, a random processX(t), t ≥ 0,\nis said to have independent incrementsif X(0) ¼ 0 and, for any t1 , t2 , ··· , tn,\nthe random variables X ðt2Þ/C0 X ðt1Þ, X ðt3Þ/

### Enhanced RAG pipeleine features

In [22]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke(prompt)
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("attention", rag_retriever, groq_llm.llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'attention'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 111.08it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Attention is a mechanism that maps a query vector and a set of key‑value pairs to an output vector. The output is a weighted sum of the values, where the weights are computed from the similarity (e.g., dot‑product) between the query and each key.
Sources: [{'source': 'attention.pdf', 'page': 2, 'score': 0.17361843585968018, 'preview': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3...'}, {'source': 'attention.pdf', 'page': 2, 'score': 0.17361843585968018, 'preview': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3...'}, {'source': 'attention.pdf', 'page': 2, 'score': 0.17361843585968018, 'preview': '3.2 Attention\nAn attention function can

In [23]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import Any, Dict

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        sources = []
        context = ""

        if not results:
            answer = "No relevant context found."
        else:
            context = "\n\n".join(doc['content'] for doc in results)
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""

            if stream:
                print("Streaming answer:")
                answer_parts = []
                for chunk in self.llm.stream(prompt):
                    if chunk.content:
                        print(chunk.content, end="", flush=True)
                        answer_parts.append(chunk.content)
                print()
                answer = "".join(answer_parts)
            else:
                answer = self.llm.invoke(prompt).content

        citations = [f"[{i + 1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        summary = None
        if summarize and sources:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary = self.llm.invoke(summary_prompt).content

        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, groq_llm.llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.62it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:


**Attention Is All You Need** is the title of the 2017 paper that introduced the Transformer architecture. It shows that a model built solely on self‑attention (without recurrence or convolution) can learn sequence‑to‑sequence tasks (e.g., machine translation) with state‑of‑the‑art performance. The key idea is that attention mechanisms alone are sufficient for modeling long‑range dependencies in data.

Final Answer: **Attention Is All You Need** is the title of the 2017 paper that introduced the Transformer architecture. It shows that a model built solely on self‑attention (without recurrence or convolution) can learn sequence‑to‑sequence tasks (e.g., machine translation) with state‑of‑the‑art performance. The key idea is that attention mechanisms alone are sufficient for modeling long‑range dependencies in data.

Citations:
[1] attention.pdf (page 2)
[2] attention.pdf (page 2)
[3] attention.pdf (page 2)
Summary: The 2017 paper **“Attention Is All You Need”** introduced the Transformer